In [0]:
numeric = spark.read.csv("/Volumes/workspace/default/bosch/train_numeric_sample.csv", header=True, inferSchema=True)
categorical = spark.read.csv("/Volumes/workspace/default/bosch/train_categorical_sample.csv", header=True, inferSchema=True)
date = spark.read.csv("/Volumes/workspace/default/bosch/train_date_sample.csv", header=True, inferSchema=True)

print("Loaded successfully")

In [0]:
print([c for c in date.columns if "L3_S32" in c or "L1_S24_F1525" in c][:20])
print([c for c in categorical.columns if "L3_S32" in c or "L1_S24_F1525" in c][:20])

In [0]:
print("Numeric:", numeric.count(), "rows,", len(numeric.columns), "columns")
print("Categorical:", categorical.count(), "rows,", len(categorical.columns), "columns")
print("Date:", date.count(), "rows,", len(date.columns), "columns")

In [0]:
numeric.groupBy("Response").count().show()

In [0]:
from pyspark.sql.functions import col, mean as spark_mean

# Fast 1-pass execution on Spark cluster
null_counts = numeric.select([
    spark_mean(col(c).isNull().cast("int")).alias(c)
    for c in numeric.columns
])

# Driverside conversion & sorting
null_pct_row = null_counts.collect()[0].asDict()
sorted_nulls = sorted(null_pct_row.items(), key=lambda x: x[1], reverse=True)

print("Top 10 columns with highest null %:")
for colname, pct in sorted_nulls[:10]:
    print(f"{colname}: {pct:.2%}")

print("\nColumns with >90% nulls:", sum(1 for _, pct in sorted_nulls if pct > 0.9))
print("Total columns:", len(sorted_nulls))

In [0]:
joined = numeric.join(categorical, "Id", "left").join(date, "Id", "left")
print("Joined row count:", joined.count())
print("Numeric row count:", numeric.count())

In [0]:
print(date.columns[:20])
print("Total date columns:", len(date.columns))

In [0]:
import re

# Extract unique station identifiers like "L0_S0", "L0_S1" from column names
station_pattern = re.compile(r"(L\d+_S\d+)_")
stations = sorted(set(
    station_pattern.match(c).group(1)
    for c in date.columns
    if c != "Id" and station_pattern.match(c)
))

print("Total unique stations:", len(stations))
print(stations[:15])

In [0]:
from pyspark.sql.functions import col, when, greatest

visited_cols = []
for station in stations:
    station_date_cols = [c for c in date.columns if c.startswith(station + "_")]
    flags = [when(col(c).isNotNull(), 1).otherwise(0) for c in station_date_cols]

    if len(flags) == 1:
        visited_expr = flags[0]
    else:
        visited_expr = greatest(*flags)

    date = date.withColumn(f"visited_{station}", visited_expr)
    visited_cols.append(f"visited_{station}")

print("Added flags:", len(visited_cols))
date.select(["Id"] + visited_cols[:5]).show(5)

In [0]:
from functools import reduce
from pyspark.sql.functions import col

date = date.withColumn(
    "total_stations_visited",
    reduce(lambda a, b: a + b, [col(f"visited_{s}") for s in stations])
)

date.select("Id", "total_stations_visited").show(10)

In [0]:
from pyspark.sql.functions import least, col

station_time_cols = []
for station in stations:
    station_date_cols = [c for c in date.columns if c.startswith(station + "_") and c.split("_")[-1].startswith("D")]
    casted_cols = [col(c).cast("double") for c in station_date_cols]

    if len(casted_cols) == 1:
        time_expr = casted_cols[0]
    else:
        time_expr = least(*casted_cols)

    date = date.withColumn(f"time_{station}", time_expr)
    station_time_cols.append(f"time_{station}")

date.select(["Id"] + station_time_cols[:5]).show(5)

In [0]:
from pyspark.sql.functions import col

# Total time span: last station timestamp minus first station timestamp
time_cols = [col(f"time_{s}") for s in stations]

date = date.withColumn("first_timestamp", least(*time_cols))
date = date.withColumn("last_timestamp", greatest(*time_cols))
date = date.withColumn("total_process_time", col("last_timestamp") - col("first_timestamp"))

date.select("Id", "first_timestamp", "last_timestamp", "total_process_time").show(10)

In [0]:
from functools import reduce
from operator import add
from pyspark.sql.functions import col

# Filter out non-measurement columns
feature_cols = [c for c in numeric.columns if c not in ("Id", "Response")]

# Build and sum null indicator expressions
null_indicators = [col(c).isNull().cast("int") for c in feature_cols]
missing_count_expr = reduce(add, null_indicators)

numeric = numeric.withColumn("missing_measurement_count", missing_count_expr)
numeric.select("Id", "Response", "missing_measurement_count").show(10)

In [0]:
numeric.groupBy("Response").avg("missing_measurement_count").show()

In [0]:
lines = sorted(set(s.split("_")[0] for s in stations))
print("Distinct lines:", lines)
print("Stations per line:")
for line in lines:
    count = sum(1 for s in stations if s.startswith(line + "_"))
    print(f"  {line}: {count} stations")

In [0]:
from functools import reduce
from operator import add
from pyspark.sql import functions as F

new_cols = {}

for line in lines:
    line_stations = [s for s in stations if s.startswith(line + "_")]
    total_in_line = len(line_stations)
    
    if total_in_line == 0:
        continue
        
    # Coalesce nulls to 0 to safeguard the addition
    visited_flags = [F.coalesce(F.col(f"visited_{s}"), F.lit(0)) for s in line_stations]
    
    # Calculate missing percentage per line
    new_cols[f"missing_pct_{line}"] = 1 - (reduce(add, visited_flags) / total_in_line)

# Apply all column additions in a single transformation pass
date = date.withColumns(new_cols)

date.select(["Id"] + list(new_cols.keys())).show(10)

In [0]:
import pyspark.sql.functions as F

# Build average expressions for all lines at once
avg_exprs = [F.avg(f"missing_pct_{l}").alias(f"avg_{l}") for l in lines]

check = date.select(["Id"] + [f"missing_pct_{l}" for l in lines]).join(
    numeric.select("Id", "Response"), "Id"
)

# Single Spark transformation pass
check.groupBy("Response").agg(*avg_exprs).show()

In [0]:
check2 = date.select("Id", "total_process_time").join(
    numeric.select("Id", "Response"), "Id"
)

check2.groupBy("Response").agg(
    F.avg("total_process_time").alias("avg_process_time"),
    F.max("total_process_time").alias("max_process_time")
).show()

In [0]:
date.select("total_process_time").summary("min", "25%", "50%", "75%", "95%", "99%", "max").show()

In [0]:
date = date.withColumn("log_total_process_time", F.log1p("total_process_time"))

date = date.withColumn(
    "process_velocity",
    F.col("total_process_time") / F.coalesce(F.col("total_stations_visited"), F.lit(1))
)

date = date.withColumn(
    "is_high_dwell",
    (F.col("total_process_time") > 43.24).cast("int")
)

date.select("Id", "total_process_time", "log_total_process_time", "process_velocity", "is_high_dwell").show(10)

In [0]:
print("Numeric columns:", len(numeric.columns))
print("Categorical columns:", len(categorical.columns))
print("Date columns:", len(date.columns))

## Phase 4: Feature Selection

### 4.1 Consolidate engineered features into modeling table

In [0]:
engineered_cols = [
    "Id",
    "total_stations_visited",
    "total_process_time",
    "log_total_process_time",
    "process_velocity",
    "is_high_dwell"
] + [f"missing_pct_{l}" for l in lines] + [f"visited_{s}" for s in stations]

model_features = date.select(engineered_cols).join(
    numeric.select("Id", "Response"), "Id"
)

print("Engineered feature table shape:", model_features.count(), len(model_features.columns))
model_features.show(5)

### 4.2 Rank raw numeric sensor columns by importance (shallow tree filter)

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier

raw_feature_cols = [c for c in numeric.columns if c not in ("Id", "Response", "missing_measurement_count")]

numeric_filled = numeric.fillna(0, subset=raw_feature_cols)

assembler = VectorAssembler(inputCols=raw_feature_cols, outputCol="features")
assembled = assembler.transform(numeric_filled)

dt = DecisionTreeClassifier(labelCol="Response", featuresCol="features", maxDepth=5)
dt_model = dt.fit(assembled)

importances = list(zip(raw_feature_cols, dt_model.featureImportances.toArray()))
importances_sorted = sorted(importances, key=lambda x: x[1], reverse=True)

print("Top 20 important raw numeric features:")
for name, score in importances_sorted[:20]:
    if score > 0:
        print(f"{name}: {score:.4f}")

###4.2 Rank raw categorical sensor columns by importance (Chi Square)

In [0]:
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import ChiSquareTest

# 0. Prepare categorical features (similar to numeric features in cell 25)
cat_feature_cols = [c for c in categorical.columns if c != "Id"]
categorical_filled = categorical.fillna("unknown", subset=cat_feature_cols)

# 1. Join once
cat_data = categorical_filled.join(numeric.select("Id", "Response"), "Id")

# 2. Hash string columns natively into numeric indices (bypasses StringIndexer Pipeline)
hash_exprs = {
    f"{c}_idx": F.coalesce(F.abs(F.hash(F.col(c))), F.lit(0)).cast("double")
    for c in cat_feature_cols
}
indexed_df = cat_data.withColumns(hash_exprs)

# 3. Assemble and run Chi-Square test globally
idx_cols = [f"{c}_idx" for c in cat_feature_cols]
assembler = VectorAssembler(inputCols=idx_cols, outputCol="features", handleInvalid="keep")
assembled_df = assembler.transform(indexed_df)

# 4. Run Chi-Square statistical test
r = ChiSquareTest.test(assembled_df, "features", "Response").head()

# 5. Extract and rank results
chisq_scores = list(zip(cat_feature_cols, r.statistics))
ranked_cat_features = sorted(chisq_scores, key=lambda x: x[1], reverse=True)

print("Top 20 Important Categorical Features (by Chi-Square Statistic):")
for name, score in ranked_cat_features[:20]:
    print(f"{name}: {score:.2f}")

In [0]:
top_numeric = [name for name, score in importances_sorted[:30]]
top_categorical = [name for name, score in ranked_cat_features[:30]]

final_features = engineered_cols + top_numeric + top_categorical
final_features = [c for c in final_features if c not in ("Id",)]  # keep Id separately

print("Total final feature count:", len(final_features))
print(final_features[:10])

In [0]:
# Clean feature lists to avoid duplicate column collisions on join
clean_top_numeric = [c for c in top_numeric if c not in ("Id", "Response")]
clean_top_cat = [c for c in top_categorical if c not in ("Id", "Response")]
clean_eng_cols = [c for c in engineered_cols if c not in ("Id", "Response")]

# Build final table
final_table = model_features.select(["Id", "Response"] + clean_eng_cols)

final_table = (
    final_table
    .join(numeric.select(["Id"] + clean_top_numeric), "Id", "inner")
    .join(categorical_filled.select(["Id"] + clean_top_cat), "Id", "inner")
)

print("Final table shape:", final_table.count(), len(final_table.columns))
final_table.show(5)

## Phase 5: Modeling
###5.1Train/test split, baseline logistic regression, main model (XGBoost/LightGBM), evaluation via MCC + PR-AUC.

In [0]:
train_df, test_df = final_table.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_df.count(), "rows |", train_df.filter("Response = 1").count(), "positive")
print("Test:", test_df.count(), "rows |", test_df.filter("Response = 1").count(), "positive")

In [0]:
train_pd = train_df.toPandas()
test_pd = test_df.toPandas()

print(train_pd.shape, test_pd.shape)
print(train_pd.dtypes.value_counts())

In [0]:
print("Object columns:", train_pd.select_dtypes(include='object').columns.tolist())
print("Datetime columns:", train_pd.select_dtypes(include='datetime64[ns]').columns.tolist())

In [0]:
print(train_pd[['L3_S32_F3851', 'L1_S24_F1525']].head(10))
print(train_pd['L3_S32_F3851'].dtype)

In [0]:
print(train_pd[['L3_S32_F3851', 'L1_S24_F1525']].dropna().head(10))

In [0]:
final_table.select("L3_S32_F3851", "L1_S24_F1525").printSchema()
final_table.select("L3_S32_F3851", "L1_S24_F1525").filter("L3_S32_F3851 IS NOT NULL").show(5)

In [0]:
raw_check = spark.read.csv("/Volumes/workspace/default/bosch/train_categorical_sample.csv", header=True, inferSchema=False)
raw_check.select("L3_S32_F3851", "L1_S24_F1525").filter("L3_S32_F3851 IS NOT NULL").show(10, truncate=False)

In [0]:
from pyspark.sql.types import StringType

categorical_fixed = spark.read.csv(
    "/Volumes/workspace/default/bosch/train_categorical_sample.csv",
    header=True,
    inferSchema=False  # force everything to load as string — no guessing
)

# Confirm fix on the two known-bad columns
categorical_fixed.select("L3_S32_F3851", "L1_S24_F1525").printSchema()

In [0]:
old_types = dict(categorical.dtypes)
new_types = dict(categorical_fixed.dtypes)

mismatches = [c for c in old_types if old_types[c] != new_types[c]]
print(f"{len(mismatches)} columns mismatched:")
print(mismatches)

In [0]:
print(categorical_fixed.count(), len(categorical_fixed.columns))

In [0]:
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import ChiSquareTest

cat_feature_cols = [c for c in categorical_fixed.columns if c != "Id"]
categorical_filled = categorical_fixed.fillna("MISSING", subset=cat_feature_cols)

cat_data = categorical_filled.join(numeric.select("Id", "Response"), "Id")

hash_exprs = {
    f"{c}_idx": F.coalesce(F.abs(F.hash(F.col(c))), F.lit(0)).cast("double")
    for c in cat_feature_cols
}
indexed_df = cat_data.withColumns(hash_exprs)

idx_cols = [f"{c}_idx" for c in cat_feature_cols]
assembler = VectorAssembler(inputCols=idx_cols, outputCol="features", handleInvalid="keep")
assembled_df = assembler.transform(indexed_df)

r = ChiSquareTest.test(assembled_df, "features", "Response").head()

chisq_scores = list(zip(cat_feature_cols, r.statistics))
ranked_cat_features_clean = sorted(chisq_scores, key=lambda x: x[1], reverse=True)

print("Top 20 Important Categorical Features (Clean Data):")
for name, score in ranked_cat_features_clean[:20]:
    print(f"{name}: {score:.2f}")

In [0]:
final_table_clean = numeric.select("Id", *[c for c in final_table.columns if c in numeric.columns and c != "Id"]) \
    .join(categorical_fixed.select("Id", *[c for c in final_table.columns if c in categorical_fixed.columns and c != "Id"]), "Id") \
    .join(date.select("Id", *[c for c in final_table.columns if c in date.columns and c != "Id"]), "Id")

print(final_table_clean.count(), len(final_table_clean.columns))

In [0]:
final_table_clean.select("L3_S32_F3851", "L1_S24_F1525").printSchema()
final_table_clean.select("L3_S32_F3851", "L1_S24_F1525").filter("L3_S32_F3851 IS NOT NULL").show(5)

In [0]:
train_df_clean, test_df_clean = final_table_clean.randomSplit([0.8, 0.2], seed=42)

train_pd = train_df_clean.toPandas()
test_pd = test_df_clean.toPandas()

print(train_pd.shape, test_pd.shape)
print(train_pd.dtypes.value_counts())

In [0]:
print("Train:", train_pd.shape[0], "rows |", train_pd['Response'].sum(), "positive")
print("Test:", test_pd.shape[0], "rows |", test_pd['Response'].sum(), "positive")

In [0]:
import pandas as pd
object_cols = train_pd.select_dtypes(include='object').columns.tolist()

for col in object_cols:
    train_pd[col] = train_pd[col].astype('category')
    test_pd[col] = test_pd[col].astype(pd.CategoricalDtype(categories=train_pd[col].cat.categories))

X_train = train_pd.drop(columns=['Id', 'Response'])
y_train = train_pd['Response']
X_test = test_pd.drop(columns=['Id', 'Response'])
y_test = test_pd['Response']

print(X_train.shape, X_test.shape)
print(X_train.dtypes.value_counts())

In [0]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

print("Train:", X_tr.shape[0], "| positive:", y_tr.sum())
print("Val:", X_val.shape[0], "| positive:", y_val.sum())

In [0]:
%pip install lightgbm

In [0]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import matthews_corrcoef, average_precision_score, classification_report

neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
scale_pos_weight = neg / pos

model = lgb.LGBMClassifier(
    objective='binary',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_estimators=300,
    learning_rate=0.05
)

model.fit(X_tr, y_tr)

# Tune threshold on VALIDATION set only
y_val_proba = model.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.1, 0.9, 81)
mcc_scores = [matthews_corrcoef(y_val, (y_val_proba >= t).astype(int)) for t in thresholds]
best_idx = np.argmax(mcc_scores)
best_threshold = thresholds[best_idx]

print(f"Optimal threshold (from validation): {best_threshold:.3f}")
print(f"Validation MCC at this threshold: {mcc_scores[best_idx]:.4f}")

In [0]:
y_test_proba = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= best_threshold).astype(int)

final_mcc = matthews_corrcoef(y_test, y_test_pred)
final_pr_auc = average_precision_score(y_test, y_test_proba)

print(f"Final Test MCC: {final_mcc:.4f}")
print(f"Final Test PR-AUC: {final_pr_auc:.4f}")
print(classification_report(y_test, y_test_pred))

In [0]:
cat_cols_tr = X_tr.select_dtypes(include='category').columns.tolist()
cat_cols_test = X_test.select_dtypes(include='category').columns.tolist()

print("In X_tr but not X_test:", set(cat_cols_tr) - set(cat_cols_test))
print("In X_test but not X_tr:", set(cat_cols_test) - set(cat_cols_tr))

for col in cat_cols_tr:
    if col in cat_cols_test:
        tr_cats = set(X_tr[col].cat.categories)
        test_cats = set(X_test[col].cat.categories)
        if tr_cats != test_cats:
            print(f"{col}: category mismatch — train has {tr_cats - test_cats}, test has {test_cats - tr_cats}")

In [0]:
print(X_test['L3_S32_F3854'].dtype)
print(X_test['L3_S32_F3851'].dtype)
print(X_test[['L3_S32_F3854', 'L3_S32_F3851']].head(10))

In [0]:
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_cols:
    all_categories = pd.concat([
        X_tr[col].astype(str),
        X_val[col].astype(str),
        X_test[col].astype(str)
    ]).unique()
    
    cat_type = pd.CategoricalDtype(categories=all_categories)
    
    X_tr[col] = X_tr[col].astype(str).astype(cat_type)
    X_val[col] = X_val[col].astype(str).astype(cat_type)
    X_test[col] = X_test[col].astype(str).astype(cat_type)

# Verify no mismatches remain
mismatch_found = False
for col in cat_cols:
    if set(X_tr[col].cat.categories) != set(X_test[col].cat.categories):
        print(f"Still mismatched: {col}")
        mismatch_found = True

print("All aligned" if not mismatch_found else "Mismatches remain")

In [0]:
model = lgb.LGBMClassifier(
    objective='binary',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_estimators=300,
    learning_rate=0.05
)

model.fit(X_tr, y_tr)

# Tune threshold on validation
y_val_proba = model.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.1, 0.9, 81)
mcc_scores = [matthews_corrcoef(y_val, (y_val_proba >= t).astype(int)) for t in thresholds]
best_idx = np.argmax(mcc_scores)
best_threshold = thresholds[best_idx]

print(f"Optimal threshold (from validation): {best_threshold:.3f}")
print(f"Validation MCC at this threshold: {mcc_scores[best_idx]:.4f}")

# Apply once to test set
y_test_proba = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= best_threshold).astype(int)

final_mcc = matthews_corrcoef(y_test, y_test_pred)
final_pr_auc = average_precision_score(y_test, y_test_proba)

print(f"\nFinal Test MCC: {final_mcc:.4f}")
print(f"Final Test PR-AUC: {final_pr_auc:.4f}")
print(classification_report(y_test, y_test_pred))

In [0]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, pr_thresholds = precision_recall_curve(y_val, y_val_proba)

# Show a few operating points
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10}")
for target_recall in [0.3, 0.4, 0.5, 0.6, 0.7]:
    idx = np.argmin(np.abs(recalls - target_recall))
    print(f"{pr_thresholds[idx]:>10.3f} {precisions[idx]:>10.3f} {recalls[idx]:>10.3f}")

In [0]:
balanced_threshold = 0.464

y_test_pred_balanced = (y_test_proba >= balanced_threshold).astype(int)

balanced_mcc = matthews_corrcoef(y_test, y_test_pred_balanced)
print(f"Test MCC at balanced threshold: {balanced_mcc:.4f}")
print(classification_report(y_test, y_test_pred_balanced))